In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Jinsaryko/Alexis", 
    repo_type="dataset", local_dir="./Alexis", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files: 100%|██████████| 1/1 [00:03<00:00,  3.49s/it]


'/home/ubuntu/Alexis'

In [3]:
files = glob('Alexis/*/*.parquet')
len(files)

1

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
# data = loop((files, 0))
# data

In [7]:
audio_files = [d['audio_filename'] for d in data]

with open('alexis-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
# !zip -rq Alexis_audio.zip Alexis_audio
# !hf upload malaysia-ai/Multilingual-TTS Alexis_audio.zip --repo-type=dataset

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Alexis_audio/Alexis-data-train-00000-of-00001_0.mp3',
 'text': 'I thought you were here about our line of work, Valentina asked Eric, turning towards him on the couch so her bare knees brushed his leg.',
 'speaker': 'Alexis_audio'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Alexis')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 647.77ba/s]
Processing Files (1 / 1): 100%|██████████|  164kB /  164kB,  821kB/s  
New Data Upload: 100%|██████████|  164kB /  164kB,  821kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.68 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f80130b75aa225e516b587a719cee222621a3abe', commit_message='Upload dataset', commit_description='', oid='f80130b75aa225e516b587a719cee222621a3abe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)